# 08. Transformer 体系

Transformer 的关键在于让序列中的每个位置直接和其他位置建立联系。

## Transformer的形式化定义

Transformer 是一种基于注意力机制的序列建模架构，其核心在于通过可学习的 Query、Key、Value 交互直接建模任意位置之间的依赖关系。与依赖递推的循环网络不同，Transformer 可以在单层内部实现全局上下文交互。

## 输入、输出与参数化方式

输入为 token 序列的嵌入表示及位置信息。每个 block 内部包含注意力子层和前馈子层，输出仍为逐 token 表示。对于 Decoder-only 模型，最终表示再映射到词表 logits，用于下一 token 预测。

## 结构分解与信息流

            典型 Transformer Block 包含：

- 多头自注意力：负责 token 间的信息交换
- 前馈网络：负责 token 内部特征变换
- 残差连接：保证深层优化稳定
- 归一化层：稳定训练动力学

多头机制使模型能够在同一层并行学习不同类型的依赖模式，而位置编码则保证模型保留序列顺序信息。

## 优化目标与训练机制

            语言建模场景中，Transformer 常以 next-token prediction 为目标，通过交叉熵训练。由于注意力机制使每个 token 都能直接访问上下文，因此模型更容易学习长程依赖和复杂语法语义结构。

在大规模预训练中，优化器、学习率调度、归一化形式与注意力实现会共同决定训练稳定性和扩展上限。

## 计算复杂度、统计性质与工程代价

            标准自注意力在序列长度维度上的复杂度为 $O(n^2)$，这既带来了强大的全局建模能力，也带来了长上下文代价高昂的问题。

因此，现代 Transformer 演化出多种工程改进，包括 RoPE、KV Cache、GQA、FlashAttention 等，用于降低推理成本和改善长上下文效率。

## 与相邻模型的差异

            与 RNN 相比，Transformer 更易并行、长依赖建模更强。
与 CNN 相比，Transformer 缺乏强局部先验，但更擅长全局关系建模。
与 MoE 结合后，Transformer 又可以进一步扩展为稀疏激活的大模型架构。

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis("off")
ax.set_xlim(0, 12)
ax.set_ylim(0, 8)

blocks = [
    (6, 7.0, 2.4, 0.8, "#4C78A8", "Token Embedding + Positional Encoding"),
    (6, 5.7, 2.2, 0.9, "#F58518", "Multi-Head Self-Attention"),
    (6, 4.4, 2.0, 0.8, "#E45756", "Residual + Norm"),
    (6, 3.1, 2.0, 0.9, "#72B7B2", "Feed-Forward Network"),
    (6, 1.8, 2.0, 0.8, "#54A24B", "Residual + Norm"),
    (6, 0.6, 1.8, 0.8, "#B279A2", "LM Head / Output"),
]
for x, y, w, h, color, text in blocks:
    rect = plt.Rectangle((x - w/2, y - h/2), w, h, facecolor=color, edgecolor="black")
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=11)
for i in range(len(blocks) - 1):
    ax.annotate("", xy=(6, blocks[i+1][1] + blocks[i+1][3]/2), xytext=(6, blocks[i][1] - blocks[i][3]/2),
                arrowprops=dict(arrowstyle="->", lw=1.8))
ax.set_title("Decoder-only Transformer Block 的层级结构")
plt.show()

## 先建立直觉

            Transformer 最重要的突破，是它不再要求信息必须一步一步传递。

在 RNN 里，如果第 1 个词想影响第 50 个词，信息必须经过 49 次传递；
但在 Transformer 里，第 50 个词可以直接“看”第 1 个词。

所以它最核心的优点是：

- 更容易建模长距离依赖
- 更适合并行计算
- 对现代硬件更友好

## 架构是怎么工作的

            Transformer Block 通常包含两大部分：

- Attention：让 token 之间彼此交流
- FFN：让每个 token 在自己的特征空间里进一步变换

多头注意力的含义也很重要：

- 一个头可能更关注位置关系
- 一个头可能更关注语义关联
- 一个头可能更关注局部搭配

你可以把多头理解成“同时用多种角度看同一句话”。

## 训练时到底发生了什么

            Transformer 的训练流程本身并不神秘，还是前向、损失、反向传播、更新参数。

真正需要理解的是：

- 它的表示能力来自 token 间的全局交互
- 它的代价也来自 token 间的全局交互

也就是说，Transformer 强的地方和贵的地方，其实是同一件事：
注意力机制。

## 什么时候该用它

            Transformer 适合：

- 文本建模
- 多模态任务
- 长依赖序列建模
- 需要大规模预训练的任务

它之所以成为大语言模型主干，不是偶然，而是因为它在表达能力、并行性和扩展性之间取得了很好的平衡。

## 最常见的误区

            常见误区：

1. **误以为 Transformer 只有 Attention**
   FFN、Norm、Residual 也同样关键。

2. **误以为注意力权重就是完整解释**
   注意力可以提供线索，但不等于完整因果解释。

3. **误以为 Transformer 不需要位置信息**
   如果没有位置编码，它根本不知道 token 的顺序。

## 1. Self-Attention 的核心公式

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]


import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)


print("临时目录:", temp_root)

In [ ]:
from collections import Counter
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split

positive_words = ["good", "great", "excellent", "helpful", "enjoyable"]
negative_words = ["bad", "terrible", "boring", "awful", "annoying"]
nouns = ["movie", "course", "service", "book", "game", "app"]

def build_samples():
    samples = []
    for noun in nouns:
        for word in positive_words:
            samples.append((f"{noun} is {word}", 1))
            samples.append((f"{noun} is very {word}", 1))
            samples.append((f"{noun} is not {word}", 0))
        for word in negative_words:
            samples.append((f"{noun} is {word}", 0))
            samples.append((f"{noun} is really {word}", 0))
            samples.append((f"{noun} is not {word}", 1))
    random.shuffle(samples)
    return samples

samples = build_samples()
texts = [text for text, label in samples]
labels = [label for text, label in samples]

In [ ]:
def tokenize(text):
    return text.split()

counter = Counter()
for text in texts:
    counter.update(tokenize(text))

vocab = {"[PAD]": 0, "[UNK]": 1, "[CLS]": 2}
for token, _ in counter.most_common():
    vocab[token] = len(vocab)

max_len = max(len(tokenize(text)) for text in texts) + 1

def encode_text(text):
    tokens = ["[CLS]"] + tokenize(text)
    ids = [vocab.get(token, vocab["[UNK]"]) for token in tokens]
    pad_len = max_len - len(ids)
    ids = ids + [vocab["[PAD]"]] * pad_len
    mask = [1] * len(tokens) + [0] * pad_len
    return ids, mask

encoded = [encode_text(text) for text in texts]
input_ids = np.array([item[0] for item in encoded], dtype=np.int64)
attention_mask = np.array([item[1] for item in encoded], dtype=np.float32)
labels = np.array(labels, dtype=np.int64)

X_train_ids, X_test_ids, X_train_mask, X_test_mask, y_train, y_test, train_texts, test_texts = train_test_split(
    input_ids, attention_mask, labels, texts, test_size=0.2, random_state=42, stratify=labels
)

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train_ids), torch.tensor(X_train_mask), torch.tensor(y_train)),
    batch_size=32,
    shuffle=True,
)
test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test_ids), torch.tensor(X_test_mask), torch.tensor(y_test)),
    batch_size=64,
    shuffle=False,
)

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=64):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]


class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4, ff_dim=128, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len=max_len + 2)
        self.attention = nn.MultiheadAttention(d_model, num_heads, dropout=0.1, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, ff_dim), nn.ReLU(), nn.Linear(ff_dim, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, input_ids, attention_mask):
        x = self.embedding(input_ids)
        x = self.pos_encoding(x)
        key_padding_mask = attention_mask == 0
        attn_out, attn_weights = self.attention(
            x, x, x, key_padding_mask=key_padding_mask, need_weights=True, average_attn_weights=False
        )
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ffn(x))
        return self.classifier(x[:, 0, :]), attn_weights


def train_transformer(model, train_loader, test_loader, epochs=25, lr=2e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}

    for _ in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for batch_ids, batch_mask, batch_y in train_loader:
            optimizer.zero_grad()
            logits, _ = model(batch_ids, batch_mask)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch_ids.size(0)
            train_correct += (logits.argmax(dim=1) == batch_y).sum().item()
            train_total += batch_ids.size(0)

        model.eval()
        test_loss, test_correct, test_total = 0.0, 0, 0
        with torch.no_grad():
            for batch_ids, batch_mask, batch_y in test_loader:
                logits, _ = model(batch_ids, batch_mask)
                loss = criterion(logits, batch_y)
                test_loss += loss.item() * batch_ids.size(0)
                test_correct += (logits.argmax(dim=1) == batch_y).sum().item()
                test_total += batch_ids.size(0)

        history["train_loss"].append(train_loss / train_total)
        history["train_acc"].append(train_correct / train_total)
        history["test_loss"].append(test_loss / test_total)
        history["test_acc"].append(test_correct / test_total)
    return history

In [ ]:
transformer = TransformerClassifier(vocab_size=len(vocab))
transformer_history = train_transformer(transformer, train_loader, test_loader, epochs=25, lr=2e-3)
print("最终测试准确率:", round(transformer_history["test_acc"][-1], 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].plot(transformer_history["train_loss"], label="train loss")
axes[0].plot(transformer_history["test_loss"], label="test loss")
axes[0].set_title("Transformer 损失曲线")
axes[0].legend()

axes[1].plot(transformer_history["train_acc"], label="train acc")
axes[1].plot(transformer_history["test_acc"], label="test acc")
axes[1].set_title("Transformer 准确率曲线")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
transformer.eval()
test_ids_tensor = torch.tensor(X_test_ids)
test_mask_tensor = torch.tensor(X_test_mask)
with torch.no_grad():
    logits, attn_weights = transformer(test_ids_tensor, test_mask_tensor)
    preds = logits.argmax(dim=1).numpy()

print("测试准确率:", round(accuracy_score(y_test, preds), 4))
plt.figure(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(y_test, preds, cmap="Blues")
plt.title("Transformer 文本分类混淆矩阵")
plt.show()

In [ ]:
candidate_idx = next(i for i, text in enumerate(test_texts) if " not " in text)
sample_text = test_texts[candidate_idx]
sample_ids = torch.tensor(X_test_ids[candidate_idx : candidate_idx + 1])
sample_mask = torch.tensor(X_test_mask[candidate_idx : candidate_idx + 1])

with torch.no_grad():
    sample_logits, sample_attn = transformer(sample_ids, sample_mask)
    sample_pred = sample_logits.argmax(dim=1).item()

valid_len = int(sample_mask.sum().item())
tokens = ["[CLS]"] + sample_text.split()
cls_attention = sample_attn[0, 0, 0, :valid_len].numpy()

plt.figure(figsize=(10, 4))
sns.heatmap(
    cls_attention.reshape(1, -1),
    annot=np.round(cls_attention.reshape(1, -1), 2),
    cmap="YlGnBu",
    xticklabels=tokens,
    yticklabels=["[CLS] query"],
)
plt.title(f"样本文本: '{sample_text}' | 预测标签: {sample_pred}")
plt.show()